In [9]:
import os
import glob
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel

In [5]:
# Device setup
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: mps


In [10]:
MODEL_NAME = "all-MiniLM-L6-v2"
# Can be either:
# 1) a single parquet file, or
# 2) a directory containing Spark-style part-* shards
INPUT_PATH = "/Users/preye/Downloads/Masters AI:ML/Big Data & ML Application/Coursework/products_semantic_export"
OUTPUT_FILE = "product_embedding_index.parquet"

In [11]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 384, padding_idx=0)
    (position_embeddings): Embedding(512, 384)
    (token_type_embeddings): Embedding(2, 384)
    (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-5): 6 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=384, out_features=384, bias=True)
            (key): Linear(in_features=384, out_features=384, bias=True)
            (value): Linear(in_features=384, out_features=384, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=384, out_features=384, bias=True)
            (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
    

In [18]:
def mean_pool(last_hidden_state, attention_mask):
    mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask_expanded, dim=1)
    summed_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
    return summed / summed_mask


def embed_texts(texts, batch_size=64):
    embeddings = []

    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]

            encoded = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=256,
                return_tensors="pt"
            )

            encoded = {k: v.to(device) for k, v in encoded.items()}
            model_output = model(**encoded)

            pooled = mean_pool(
                model_output.last_hidden_state,
                encoded["attention_mask"]
            )

            pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
            embeddings.append(pooled.cpu())

    return torch.cat(embeddings)

In [13]:
def load_products(path: str) -> pd.DataFrame:
    """Load products from a parquet file or Spark-style part-* shard directory."""
    path = os.path.expanduser(path)

    if os.path.isdir(path):
        part_files = sorted(glob.glob(os.path.join(path, "part-*")))
        if not part_files:
            raise FileNotFoundError(f"No part-* files found in directory: {path}")
    elif os.path.isfile(path):
        if os.path.basename(path).startswith("part-"):
            part_files = sorted(glob.glob(os.path.join(os.path.dirname(path), "part-*")))
            if not part_files:
                part_files = [path]
        else:
            part_files = [path]
    else:
        raise FileNotFoundError(
            f"Input path not found: {path}\n"
            f"Current working directory: {os.getcwd()}"
        )

    frames = []
    for file_path in part_files:
        if file_path.endswith(".parquet") or ".snappy" in file_path:
            frames.append(pd.read_parquet(file_path))
        else:
            frames.append(pd.read_csv(file_path, escapechar="\\", encoding="utf-8"))

    return pd.concat(frames, ignore_index=True)


def main():
    df = load_products(INPUT_PATH)

    df = df[["parent_asin", "title", "product_text"]].copy()
    df["product_text"] = df["product_text"].fillna("")

    print("Total products:", len(df))

    texts = df["product_text"].tolist()
    embeddings = embed_texts(texts)

    # Convert embeddings to list for storage
    df["embedding"] = embeddings.numpy().tolist()

    df[["parent_asin", "title", "embedding"]].to_parquet(OUTPUT_FILE, index=False)

    print("Saved embedding index:", OUTPUT_FILE)


if __name__ == "__main__":
    main()

Total products: 1610012
Saved embedding index: product_embedding_index.parquet


In [22]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Load saved embeddings
df = pd.read_parquet("product_embedding_index.parquet")

# Convert embeddings column back to matrix
product_embeddings = np.vstack(df["embedding"].values)

# Define query
query = "wireless headphones for studying"

# Embed query
query_embedding = embed_texts([query]).numpy()

# Compute similarity
scores = cosine_similarity(query_embedding, product_embeddings)[0]

# Rank
df["semantic_score"] = scores
results = df.sort_values("semantic_score", ascending=False)

results[["parent_asin", "title", "semantic_score"]].head(10)

,parent_asin,title,semantic_score
575516,B09B76WQDT,"Headphones Wireless Bluetooth, Over Ear Headph...",0.660529
379471,B09C3SMBPF,Kids Bluetooth Headphones Over Ear with Microp...,0.659650
883925,B0B648TKMC,"eKids That Girl Lay Lay Bluetooth Headphones, ...",0.659100
974896,B09LGYDQQ8,"Wireless Kids Headphones with Microphones, Wir...",0.651515
1403695,B0C53XKQHM,48 Pack Classroom Headphones on Ear Wired Ster...,0.637287
1530596,B09MLN2Q2Q,"TECHSHARE Wireless Bluetooth Headset, Unicorns...",0.636808
677946,B092LPZYCS,"ONANOFF BuddyPhones Play+, Wireless Bluetooth ...",0.636529
871212,B07TFDKKHV,Comfortable Portable Wireless Headphones Hi-fi...,0.636462
714007,B08ZDMKBZK,Student Wireless Bluetooth Headphones Suitable...,0.635070
257247,B0921WYS7Q,Kids Headphones for School (3 Pack) - On-Ear P...,0.634405


In [23]:
k = 10
top_k_idx = np.argsort(scores)[-k:][::-1]

top_results = df.iloc[top_k_idx]

In [25]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# load once
df = pd.read_parquet("product_embedding_index.parquet")
product_embeddings = np.vstack(df["embedding"].values)

def retrieve_top_k(query: str, k: int = 10):
    query_embedding = embed_texts([query]).numpy()
    scores = cosine_similarity(query_embedding, product_embeddings)[0]

    temp = df.copy()
    temp["semantic_score"] = scores

    results = temp.sort_values("semantic_score", ascending=False).head(k)
    return results[["parent_asin", "title", "semantic_score"]]

In [27]:
# retrieve_top_k("wireless headphones for studying", k=10)
# retrieve_top_k("good mouse for video editing", k=10)
retrieve_top_k("Curved monitor for gaming", k=100)

,parent_asin,title,semantic_score
251128,B01MA2B2KU,SAMSUNG CFG70 Series 27-Inch 1ms Curved Gaming...,0.719397
1507489,B095J7DD87,"Deco Gear 49"" Curved Ultrawide E-LED Gaming Mo...",0.710102
1402034,B0C5XLGPPH,XGaming 27 Inch Curved Gaming Monitor 144Hz/16...,0.708763
985751,B09ZNW2ZC1,CRUA 24 inch 144hz/180hz Curved Gaming Monitor...,0.697843
1003117,B08GC68YKJ,"Oecrayy 24"" Curved Gaming Monitor 75Hz 1080p 2...",0.695501
1555388,B091NC4VRF,Z-Edge 30-inch Curved Gaming Monitor 21:9 2560...,0.691261
85189,B076B1WKVD,Viotek 144Hz GN32LD QHD 32 inch Curved Gaming ...,0.689602
728199,B01FXDVZF2,"Lenovo Monitor, Y27g 27-Inch Curved Gaming Mon...",0.683266
803351,B09VDK9WZQ,CRUA 21.5 Inch Curved Computer Monitor Full HD...,0.680772
1340566,B08CPQ4ZL6,HP X24c Gaming Monitor | 1500R Curved Gaming M...,0.680631


In [29]:
query = "wireless headphones for studying"
semantic_results = retrieve_top_k(query, k=100)
semantic_results.to_parquet("semantic_results.parquet", index=False)